# Train GPT-BERT on native (non-translated) Hindi/Telugu data

Trains monolingual GPT-BERT models from scratch on the CC-100-derived native data (`pulipakav-1/hi-te`), using this project's existing `babybabellm-gptbert` training code (`Babylm2026/gpt-bert/{hindi,telugu}`), with the exact hyperparameters used for the original cluster run (`global_batch_size=32768`, `local_batch_size=128`, `max_steps=15625` -- see `scripts/train_model.sh`). The original ran across 3-4 GPUs in parallel; on a single Colab GPU the same total compute happens sequentially, so expect this to take roughly 3-4x longer than the original cluster run did.

**Each step is its own cell, looping over both Hindi and Telugu before moving to the next step** -- run all cells top to bottom.

Every `!python` call below is followed by an explicit exit-code check that raises an error and stops the cell if that step failed -- `!` commands don't do this on their own in Jupyter, which caused a real problem earlier (an interrupted step let training run anyway and fail silently).

In [ ]:
# Cell 1: clone the repo (idempotent -- skips cloning if BabyLM already exists) and
# install dependencies for both languages
import os

if not os.path.isdir("/content/BabyLM"):
    !git clone https://github.com/vishnup22/BabyLM.git /content/BabyLM

%cd /content/BabyLM
!git checkout evaluation
!git pull
!pip install -q -r Babylm2026/gpt-bert/hindi/requirements.txt
!pip install -q -r Babylm2026/gpt-bert/telugu/requirements.txt

REPO_ROOT = "/content/BabyLM"
LANGS = ["hindi", "telugu"]
LANG_CODE = {"hindi": "hi", "telugu": "te"}

In [ ]:
# Cell 2: log in to Hugging Face (only needed if you want the optional push-to-HF step at
# the end -- pulipakav-1/hi-te itself is a public dataset, no token needed just to read it)
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # use whatever secret name you saved your token under
!hf auth whoami

In [ ]:
# Cell 3: hyperparameters -- match Babylm2026/gpt-bert/hindi/scripts/train_model.sh exactly
MAX_STEPS = 15625
GLOBAL_BATCH_SIZE = 32768
LOCAL_BATCH_SIZE = 128  # lower this if you hit an out-of-memory error
SEQ_LENGTH = 128

In [ ]:
# Cell 4: download the native data for both languages and lay it out where the
# tokenizer/shard tools expect it: data/raw/<dataset_name>/<dataset_name>.train.<lang>.txt
from pathlib import Path
from huggingface_hub import hf_hub_download

for LANG in LANGS:
    lang_code = LANG_CODE[LANG]
    dataset_name = f"native-{LANG}"
    lang_dir = Path(REPO_ROOT) / "Babylm2026" / "gpt-bert" / LANG

    src_path = hf_hub_download(repo_id="pulipakav-1/hi-te", filename=f"{LANG}.txt", repo_type="dataset")
    raw_dir = lang_dir / "data" / "raw" / dataset_name
    raw_dir.mkdir(parents=True, exist_ok=True)
    dest_path = raw_dir / f"{dataset_name}.train.{lang_code}.txt"
    dest_path.write_bytes(Path(src_path).read_bytes())
    print(f"[{LANG}] Placed {dest_path} ({dest_path.stat().st_size:,} bytes)")

In [ ]:
# Cell 5: train a fresh tokenizer on each language's native data (vocab_size=16384) --
# not reusing the old translated-data tokenizer, which would reintroduce the
# tokenizer-granularity issue found earlier in this project
for LANG in LANGS:
    dataset_name = f"native-{LANG}"
    %cd {REPO_ROOT}/Babylm2026/gpt-bert/{LANG}
    !python tools/train_tokenizer_local.py \
      --dataset {dataset_name} \
      --data_root data/raw \
      --output tokenizers/tokenizer_native_16384.json \
      --vocab_size 16384
    assert _exit_code == 0, f"[{LANG}] tokenizer training failed (exit code {_exit_code})"

In [ ]:
# Cell 6: tokenize and write train/valid shards (2% held out for validation) for both
# languages
for LANG in LANGS:
    dataset_name = f"native-{LANG}"
    %cd {REPO_ROOT}/Babylm2026/gpt-bert/{LANG}
    !python tools/prepare_local_shards.py \
      --dataset {dataset_name} \
      --data_root data/raw \
      --tokenizer tokenizers/tokenizer_native_16384.json \
      --output_base data/processed \
      --valid_fraction 0.02
    assert _exit_code == 0, f"[{LANG}] shard preparation failed (exit code {_exit_code})"

In [ ]:
# Cell 7: train, single GPU, one language after the other
os.environ["WANDB_MODE"] = "disabled"  # no W&B account needed
os.environ["PYTHONUNBUFFERED"] = "1"   # keep progress output streaming live

for LANG in LANGS:
    print(f"\n{'=' * 70}\nTraining {LANG}\n{'=' * 70}")
    %cd {REPO_ROOT}/Babylm2026/gpt-bert/{LANG}/pretraining
    !python train_single_gpu.py \
      --train_path ../data/processed/train \
      --valid_path ../data/processed/valid \
      --config_file ../configs/base.json \
      --tokenizer_path ../tokenizers/tokenizer_native_16384.json \
      --name native-{LANG}-gptbert \
      --output_dir ../model_checkpoints \
      --hybrid_numerator 2 \
      --hybrid_denominator 3 \
      --global_batch_size {GLOBAL_BATCH_SIZE} \
      --local_batch_size {LOCAL_BATCH_SIZE} \
      --seq_length {SEQ_LENGTH} \
      --max_steps {MAX_STEPS} \
      --save_every 1000 \
      --validate_every 0 \
      --seed 42
    assert _exit_code == 0, f"[{LANG}] training failed (exit code {_exit_code})"
    print(f"\nFinished training {LANG}.")

## Optional: push both trained checkpoints to Hugging Face

Requires the HF login cell above to have been run with a write-scoped token.

In [ ]:
# Cell 8 (optional): push the checkpoint + tokenizer + config for both languages
from huggingface_hub import HfApi

api = HfApi()
for LANG in LANGS:
    lang_dir = f"{REPO_ROOT}/Babylm2026/gpt-bert/{LANG}"
    repo_id = f"pulipakav-1/native-{LANG}-gptbert"
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    for local_path, repo_path in [
        (f"{lang_dir}/model_checkpoints/native-{LANG}-gptbert_2_3_ema.bin", "model_ema.bin"),
        (f"{lang_dir}/tokenizers/tokenizer_native_16384.json", "tokenizer.json"),
        (f"{lang_dir}/configs/base.json", "config_base.json"),
    ]:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=repo_path, repo_id=repo_id, repo_type="model")
    print(f"[{LANG}] Pushed to https://huggingface.co/{repo_id}")